# Lab 02 — Tiền xử lý văn bản tiếng Anh
**Môn Xử lý ngôn ngữ tự nhiên | Nhóm Ba Sáu**

Dựng lại pipeline trong báo cáo `lab02_report.docx` (câu "Natural Language Processing…"): chữ thường → làm sạch → tách từ → loại stopword → gắn nhãn từ loại → stemming và lemmatization.
Kết quả kỳ vọng (ghi trong các cell) lấy từ ảnh chụp trong báo cáo.

In [ ]:
# ── Cell 1: Cài đặt thư viện (Colab) ──────────────────────────────────────
!pip install nltk

In [1]:
# ── Cell 2: Import và tải dữ liệu NLTK ────────────────────────────────────
import re
import nltk

nltk.download('punkt_tab')                    # tách từ
nltk.download('stopwords')                    # danh sách stopword
nltk.download('wordnet')                      # từ điển cho lemmatizer
nltk.download('averaged_perceptron_tagger_eng')   # bộ gắn nhãn từ loại tiếng Anh

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


True

In [2]:
# ── Cell 3: Câu mẫu ───────────────────────────────────────────────────────
text_en = "Natural Language Processing (NLP) has advanced significantly and now plays an important role in multiple real-world applications like chatbots, search engines and sentiment analysis."
text_en

'Natural Language Processing (NLP) has advanced significantly and now plays an important role in multiple real-world applications like chatbots, search engines and sentiment analysis.'

## 1. Chữ thường
Đưa mọi chữ về chữ thường để "Natural" và "natural" được xem là một từ.

In [3]:
# ── Cell 4: Chuyển chữ thường ─────────────────────────────────────────────
text_lower = text_en.lower()
print(text_lower)

natural language processing (nlp) has advanced significantly and now plays an important role in multiple real-world applications like chatbots, search engines and sentiment analysis.


## 2. Làm sạch ký tự
Giữ chữ cái a–z và khoảng trắng, bỏ dấu câu, số, ký tự đặc biệt. Lưu ý: xoá dấu gạch ngang làm "real-world" thành "realworld" (dính hai từ). Nếu muốn tách thành "real world" thì thay bằng khoảng trắng.

In [4]:
# ── Cell 5: Làm sạch bằng regex ───────────────────────────────────────────
text_clean = re.sub(r'[^a-z\s]', '', text_lower)
print(text_clean)

natural language processing nlp has advanced significantly and now plays an important role in multiple realworld applications like chatbots search engines and sentiment analysis


In [5]:
# ── Cell 6: Biến thể: thay ký tự lạ bằng khoảng trắng ─────────────────────
text_clean_sp = re.sub(r'\s+', ' ', re.sub(r'[^a-z\s]', ' ', text_lower)).strip()
print(text_clean_sp)   # "real world" được giữ thành hai từ

natural language processing nlp has advanced significantly and now plays an important role in multiple real world applications like chatbots search engines and sentiment analysis


## 3. Tách từ

In [6]:
# ── Cell 7: Tách từ bằng word_tokenize ────────────────────────────────────
from nltk.tokenize import word_tokenize

tokens = word_tokenize(text_clean)
tokens

['natural',
 'language',
 'processing',
 'nlp',
 'has',
 'advanced',
 'significantly',
 'and',
 'now',
 'plays',
 'an',
 'important',
 'role',
 'in',
 'multiple',
 'realworld',
 'applications',
 'like',
 'chatbots',
 'search',
 'engines',
 'and',
 'sentiment',
 'analysis']

## 4. Loại stopword
Dùng `set` để kiểm tra thuộc về nhanh (O(1)), thay vì `list` (O(n)) khi danh sách lớn.

In [7]:
# ── Cell 8: Loại stopword ─────────────────────────────────────────────────
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
tokens_nostop = [w for w in tokens if w not in stop_words]
print('Bị loại:', [w for w in tokens if w in stop_words])   # has, and, now, an, in, and (từ 'and' xuất hiện 2 lần)
tokens_nostop

Bị loại: ['has', 'and', 'now', 'an', 'in', 'and']


['natural',
 'language',
 'processing',
 'nlp',
 'advanced',
 'significantly',
 'plays',
 'important',
 'role',
 'multiple',
 'realworld',
 'applications',
 'like',
 'chatbots',
 'search',
 'engines',
 'sentiment',
 'analysis']

## 5. Gắn nhãn từ loại (POS tagging)
Gán nhãn ngữ pháp (NN danh từ, VB động từ, JJ tính từ, RB phó từ...) cho từng từ. Báo cáo gắn nhãn **sau** khi loại stopword; bên dưới so sánh với gắn nhãn trên câu đầy đủ, vì bộ gắn nhãn dựa vào ngữ cảnh nên bỏ stopword đi có thể làm nhãn kém chính xác hơn.

In [8]:
# ── Cell 9: POS tagging sau khi loại stopword (như báo cáo) ───────────────
nltk.pos_tag(tokens_nostop)

[('natural', 'JJ'),
 ('language', 'NN'),
 ('processing', 'NN'),
 ('nlp', 'RB'),
 ('advanced', 'VBD'),
 ('significantly', 'RB'),
 ('plays', 'VBZ'),
 ('important', 'JJ'),
 ('role', 'NN'),
 ('multiple', 'JJ'),
 ('realworld', 'NN'),
 ('applications', 'NNS'),
 ('like', 'IN'),
 ('chatbots', 'NNS'),
 ('search', 'VBP'),
 ('engines', 'JJ'),
 ('sentiment', 'NN'),
 ('analysis', 'NN')]

In [9]:
# ── Cell 10: POS tagging trên câu đầy đủ (so sánh) ────────────────────────
tagged_full = nltk.pos_tag(tokens)
tagged_full

[('natural', 'JJ'),
 ('language', 'NN'),
 ('processing', 'NN'),
 ('nlp', 'NN'),
 ('has', 'VBZ'),
 ('advanced', 'VBN'),
 ('significantly', 'RB'),
 ('and', 'CC'),
 ('now', 'RB'),
 ('plays', 'VBZ'),
 ('an', 'DT'),
 ('important', 'JJ'),
 ('role', 'NN'),
 ('in', 'IN'),
 ('multiple', 'JJ'),
 ('realworld', 'NN'),
 ('applications', 'NNS'),
 ('like', 'IN'),
 ('chatbots', 'NNS'),
 ('search', 'VBP'),
 ('engines', 'NNS'),
 ('and', 'CC'),
 ('sentiment', 'NN'),
 ('analysis', 'NN')]

## 6. Stemming và lemmatization
- **Stemming (PorterStemmer):** cắt hậu tố theo luật, nhanh nhưng có thể ra chuỗi không phải từ ("advanced" → "advanc").
- **Lemmatization (WordNetLemmatizer):** tra từ điển, ra dạng gốc có nghĩa, và cần biết **từ loại** để chính xác.

In [10]:
# ── Cell 11: Stemming ─────────────────────────────────────────────────────
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stems = [stemmer.stem(w) for w in tokens_nostop]
stems

['natur',
 'languag',
 'process',
 'nlp',
 'advanc',
 'significantli',
 'play',
 'import',
 'role',
 'multipl',
 'realworld',
 'applic',
 'like',
 'chatbot',
 'search',
 'engin',
 'sentiment',
 'analysi']

In [11]:
# ── Cell 12: Lemmatization (không chỉ định từ loại) ───────────────────────
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
lemmas = [lemmatizer.lemmatize(w) for w in tokens_nostop]   # mặc định coi mọi từ là danh từ
lemmas

['natural',
 'language',
 'processing',
 'nlp',
 'advanced',
 'significantly',
 'play',
 'important',
 'role',
 'multiple',
 'realworld',
 'application',
 'like',
 'chatbots',
 'search',
 'engine',
 'sentiment',
 'analysis']

In [12]:
# ── Cell 13: Lemmatization theo từ loại ───────────────────────────────────
from nltk.corpus import wordnet

def to_wordnet_pos(tag):
    """Đổi nhãn Penn Treebank (JJ, NN, VB, RB...) sang nhãn của WordNet."""
    return {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}.get(tag[0], wordnet.NOUN)

tagged = nltk.pos_tag(tokens_nostop)
lemmas_pos = [lemmatizer.lemmatize(w, to_wordnet_pos(t)) for w, t in tagged]
for w, a, b in zip(tokens_nostop, lemmas, lemmas_pos):
    print(f'{w:<14} mặc định: {a:<14} theo từ loại: {b}')

natural        mặc định: natural        theo từ loại: natural
language       mặc định: language       theo từ loại: language
processing     mặc định: processing     theo từ loại: processing
nlp            mặc định: nlp            theo từ loại: nlp
advanced       mặc định: advanced       theo từ loại: advance
significantly  mặc định: significantly  theo từ loại: significantly
plays          mặc định: play           theo từ loại: play
important      mặc định: important      theo từ loại: important
role           mặc định: role           theo từ loại: role
multiple       mặc định: multiple       theo từ loại: multiple
realworld      mặc định: realworld      theo từ loại: realworld
applications   mặc định: application    theo từ loại: application
like           mặc định: like           theo từ loại: like
chatbots       mặc định: chatbots       theo từ loại: chatbots
search         mặc định: search         theo từ loại: search
engines        mặc định: engine         theo từ loại: engines
se

## Tổng kết
| Bước | Kết quả (theo báo cáo) |
|---|---|
| Chữ thường + làm sạch | `natural language processing nlp has advanced ... sentiment analysis` |
| Tách từ | 24 token, gồm `has`, `and`, `now`, `an`, `in` |
| Loại stopword | còn 18 token: `natural, language, processing, nlp, advanced, ..., analysis` |
| Stemming | `natur, languag, process, nlp, advanc, significantli, play, ...` |
| Lemmatization | `natural, language, processing, nlp, advanced, significantly, play, ..., application, ...` |

**Kiến thức cần nhớ:** stemming nhanh nhưng có thể ra từ không có nghĩa; lemmatization tra từ điển và cần từ loại (`pos='v'`) để ra đúng dạng gốc; pipeline tiếng Anh cần lemmatization còn tiếng Việt cần tách từ.